# C02 — Photonic crystal slab

## Mathematical and physical model

The structure is nonmagnetic and periodic in the transverse plane. In every constant-$z$ slice the electric field satisfies
$$
\nabla\times\nabla\times\mathbf E-k_0^2\epsilon_r(\mathbf r)\mathbf E=0,\qquad
\mathbf E_t(\mathbf r)=\sum_{m,n}\mathbf e_{mn}(z)e^{i(\mathbf k_{\parallel,0}+\mathbf G_{mn})\cdot (x,y)},\qquad
\mathbf G_{mn}=2\pi(m/P_x,n/P_y).
$$
Match tangential $\mathbf E$ and $\mathbf H$ across the layer interfaces. In each exterior medium $j$, the outgoing order is propagating when
$k_{z,j,mn}^2=k_0^2\epsilon_j-|\mathbf k_{\parallel,0}+\mathbf G_{mn}|^2>0$.
The comparison uses the complex amplitudes normalized so $|a_{mn}^{q\leftarrow p}|^2$ is the incident-power fraction in an outgoing polarization channel. A one-dimensional grating retains $n=0$ even when the incident $k_y$ is nonzero.

A square lattice of holes couples external plane waves to leaky slab modes. Locally, interference between one mode and direct transmission has the form
$$
t(f)\approx t_{\mathrm{bg}}(f)+\frac{b}{f-f_0+i\gamma},\qquad f=P_x/\lambda_0 .
$$
The expression describes the local line shape near one isolated resonance. At normal incidence fourfold symmetry requires the same total spectrum for the two incident linear polarizations. Compare the resonance locations as well as transmission at fixed frequencies.

## Physical scenario

A dielectric slab contains a square lattice of air holes. The periodic modulation couples normally incident light to slab modes that can leak back into free space. Their interference with direct transmission produces narrow spectral features, so comparisons near resonance depend on both spectral sampling and numerical refinement.

## Possible applications

Guided resonances in photonic crystal slabs are used in narrowband filters, wavelength-selective reflectors, and refractive-index sensors. [Ansys photonic crystal slab benchmark](https://optics.ansys.com/hc/en-us/articles/4416343906067-Photonic-Crystal-Slab-RCWA)

## Benchmark objective

Reproduce the guided-resonance spectrum from the [S4 paper](https://web.stanford.edu/group/fan/publication/Liu_ComputerPhysicsCommunications_183_2233_2012.pdf) and the [Ansys example](https://optics.ansys.com/hc/en-us/articles/4416343906067-Photonic-Crystal-Slab-RCWA). Compare the transmission peaks near normalized frequencies 0.5058, 0.5260 and 0.5420.

## Reproduce

Use the project environment and select **Run All**. The notebook and CLI use [case.yaml](case.yaml). The native [Lumerical model](model.fsp) and [reconstruction script](model.lsf) correspond to quick-sweep point 0.

[Published comparisons and figures](../../../BENCHMARK_REPORT.md#c02) · [Periodic diagnosis](../../../PERIODIC_DIAGNOSIS.md#c02)

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

from rcwa_cv.case_loader import catalog
from rcwa_cv.notebook import case_setup, show_results, show_versions, solve_sweep
from rcwa_cv.reporting.plots import geometry_figure

case_path = catalog("C02")[0]
samples, settings, metadata = case_setup(case_path)
results = {}

print(f"{len(samples)} points, Fourier truncation index {settings.order}")
figure = geometry_figure(case_path)
display(figure)
plt.close(figure)

## meent · NumPy CPU

In [ ]:
from rcwa_cv.solvers.meent_cpu import probe, solve

print(probe())
results["meent"] = solve_sweep(solve, samples, settings)

## torcwa · CUDA GPU, complex128

In [ ]:
from rcwa_cv.solvers.torcwa_cuda import probe, solve

print(probe())
results["torcwa"] = solve_sweep(solve, samples, settings)

## grcwa · NumPy CPU

In [ ]:
from rcwa_cv.solvers.grcwa_cpu import probe, solve

print(probe())
results["grcwa"] = solve_sweep(solve, samples, settings)

## Lumerical RCWA

This cell uses the installed, licensed FDTD application through its Python API. It reuses one session
across the sweep and evaluates the same model generator used by the reconstruction script.

In [ ]:
from rcwa_cv.solvers.lumerical import session, solve

with session() as fdtd:
    print(f"Lumerical build {fdtd.version()}")
    results["lumerical"] = solve_sweep(solve, samples, settings)

## Compare with Lumerical

The spectra show total reflection, transmission and the co-polarized transmission phase.
The order plot uses a shared excitation point. The table and difference plots compare each open-source solver with Lumerical.
Periodic quick sweeps provide preliminary comparisons.

In [ ]:
show_versions(results)
show_results(metadata, results)